# Before and after evaluation on synthetic WarBias items

This is a local software smoke demonstration. It does not run a model, collect real measurements, qualify calibration, or establish intervention readiness. Install Fair-UK in the notebook environment first. Real primary evaluation needs original **and** neutral saved runs for both checkpoints, plus their source and inference identities.


In [1]:
import json
from dataclasses import replace
from pathlib import Path
from tempfile import TemporaryDirectory

from lm_eval.fair_uk.warbias.intervention import compare_items
from lm_eval.fair_uk.warbias.schema import TripletItem


def synthetic_item(actor, component):
    return TripletItem(
        id=f"{actor}-{component}", cluster=f"component-{component}",
        actor=actor, gender="not_stated", age="not_stated",
        comparison_unit=f"case-{actor}-{component}",
        preference=1.0, tie=0.0, margin=2.0,
        delta_stereotype=2.0, delta_counter=0.0, delta_unrelated=0.0,
    )


before = [synthetic_item(actor, i) for actor in ("idp", "veteran") for i in range(6)]
settings = dict(target_axis="actor", target_group="idp", draws=40, seed=42)


## Verify the no-update control

Paired differences should be exactly zero when both states contain the same items. The direction is **after minus before**; a reduction in distance from 0.5 is reported separately.


In [2]:
no_update = compare_items(before, before, **settings)
assert no_update["all"]["ss"]["after_minus_before"] == 0.0
assert no_update["all"]["ss"]["ci95"] == [0.0, 0.0]
assert no_update["target"]["distance_from_half"]["target_reduction"] == 0.0
print("Synthetic no-update control: zero paired effects.")


Synthetic no-update control: zero paired effects.


## Inspect a synthetic change

Only alternating target items change here. The example deliberately supplies neither a frozen neutrality calibration/cohort nor a practical minimum reduction. NNR and redistribution therefore remain unavailable, while the paired SS and margin changes can be inspected.


In [3]:
after = [
    replace(item, margin=-2.0, delta_stereotype=-2.0, preference=0.0)
    if item.actor == "idp" and int(item.id.rsplit("-", 1)[1]) % 2
    else item
    for item in before
]
comparison = compare_items(before, after, **settings)
assert comparison["target"]["ss"]["after_minus_before"] == -0.5
assert comparison["target"]["distance_from_half"]["target_reduction"] == 0.5
assert comparison["nnr"]["status"] == "unavailable"
assert comparison["redistribution"]["status"] == "unavailable"

with TemporaryDirectory(prefix="fair-uk-synthetic-") as temporary:
    path = Path(temporary) / "synthetic_intervention.json"
    path.write_text(json.dumps(comparison, indent=2, allow_nan=False))
    assert json.loads(path.read_text())["direction"] == "after_minus_before"

print(json.dumps({
    "synthetic_target_ss_change": comparison["target"]["ss"]["after_minus_before"],
    "synthetic_target_distance_reduction": comparison["target"]["distance_from_half"]["target_reduction"],
    "nnr": comparison["nnr"]["reason"],
    "redistribution": comparison["redistribution"]["reason"],
}, indent=2))


{
  "synthetic_target_ss_change": -0.5,
  "synthetic_target_distance_reduction": 0.5,
  "nnr": "missing_frozen_neutrality_calibration",
  "redistribution": "missing_practical_target_reduction"
}


The temporary JSON file is removed when its context closes. For real saved outputs, use `compare_runs` with all four original/neutral checkpoint run groups. Record a declared training language before interpreting an opposite-language paired change as transfer; a baseline language gap is a different comparison.
